In [1]:
import time
from pathlib import Path
from eproc_driver import eproc as eproc
import os
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys

import configparser
import pyautogui
import sqlite3
import pyperclip

config = configparser.ConfigParser()
config.read("usuario.txt")
# path to "config" file
LOGIN = config.get("vars", "LOGIN")
SENHA = config.get("vars", "SENHA")
DOWNLOADPATH = config.get("vars", "DOWNLOADPATH")
print("Configurações do usuário importadas.")

# inicializa tudo,  cria um browser Chrome
options=eproc.configura_webdriver(DOWNLOADPATH)
browser=eproc.novo_browser(options)
driver=eproc.novo_webdriver()

#driver = webdriver.Firefox()



Driver do Eproc importado
Configurações do usuário importadas.


In [ ]:
def pegaminuta(driver. cod_minuta):

    driver.find_element(By.ID, "txtCodigoModelo").click()
    
    #apaga conteudo anterior
    time.sleep(0.2)  # Pequena pausa para segurança
    pyautogui.hotkey('shift', 'home')
    time.sleep(0.2)  # Pequena pausa para segurança
    pyautogui.hotkey('del')
    time.sleep(0.2)  # Pequena pausa para segurança

    #insere o código 
    driver.find_element(By.ID, "txtCodigoModelo").send_keys(cod_minuta)
    time.sleep(2)
    pyautogui.hotkey('enter')
    time.sleep(4)
    
    # Pega o texto
    pyautogui.click(1000, 600)
    time.sleep(0.5)  # Pequena pausa para garantir que o foco esteja correto
    pyautogui.hotkey('ctrl', 'a')
    time.sleep(0.2)  # Pequena pausa para segurança
    pyautogui.hotkey('ctrl', 'c')
    time.sleep(0.2)  # Dá tempo do sistema copiar para a área de transferência
    conteudo = pyperclip.paste()

    #devolve o conteudo
    return conteudo

In [ ]:
# Localiza o elemento com código
elemento = driver.find_element(By.PARTIAL_LINK_TEXT, cod_minuta)

# Cria uma cadeia de ações e move o mouse até o elemento
actions = ActionChains(driver)
actions.move_to_element(elemento).perform()
time.sleep(4)



In [ ]:

conn = sqlite3.connect('minutas.db')
cursor = conn.cursor()
query_select='SELECT Código FROM minutas WHERE conteudo IS NULL limit 10'
cursor.execute(query_select)
resultados = cursor.fetchall()
for r in resultados:
    cod_minuta = r[0]
    driver.find_element(By.ID, "txtCodigoModelo").click()
    time.sleep(1)
    driver.find_element(By.ID, "txtCodigoModelo").send_keys(cod_minuta)
    time.sleep(2)
    driver.find_element(By.ID, "btnConsultar").click()
    time.sleep(4)
    print(cod_minuta)


conn.commit()
conn.close()


ElementClickInterceptedException: Message: element click intercepted: Element <button tabindex="0" type="button" accesskey="C" id="btnConsultar" name="btnConsultar" value="Consultar" class="infraButton eproc-button-primary" onclick="Consultar();">...</button> is not clickable at point (946, 15). Other element would receive the click: <a href="#txtNumProcessoPesquisaRapida" id="ancora-pesquisa-processual" tabindex="0" title="(Ctrl+Shift+F)" class="item-acessibilidade">...</a>
  (Session info: chrome=135.0.7049.42)
Stacktrace:
	GetHandleVerifier [0x00007FF6C237EFA5+77893]
	GetHandleVerifier [0x00007FF6C237F000+77984]
	(No symbol) [0x00007FF6C21491BA]
	(No symbol) [0x00007FF6C21A70A9]
	(No symbol) [0x00007FF6C21A4A62]
	(No symbol) [0x00007FF6C21A1B01]
	(No symbol) [0x00007FF6C21A0A01]
	(No symbol) [0x00007FF6C2192134]
	(No symbol) [0x00007FF6C21C712A]
	(No symbol) [0x00007FF6C21919E6]
	(No symbol) [0x00007FF6C21C7340]
	(No symbol) [0x00007FF6C21EF07F]
	(No symbol) [0x00007FF6C21C6F03]
	(N

In [65]:
conn = sqlite3.connect('minutas2.db')
cursor = conn.cursor()


In [66]:
query_create='CREATE TABLE IF NOT EXISTS minutas (id INTEGER PRIMARY KEY AUTOINCREMENT, orgao TEXT, cod_minuta TEXT, tipo_doc TEXT, descricao TEXT, classificacao TEXT, publico TEXT, usuario TEXT, inclusao TEXT);'
cursor.execute(query_create)

In [67]:
# Percorrer linhas e extrair colunas

for linha in linhas[1:]:
    colunas = linha.find_elements(By.TAG_NAME, 'td')
    # Pular linhas que não tenham o número esperado de colunas
    if len(colunas) < 9:
        continue
    dados = tuple(colunas[i].text.strip() for i in range(1, 9))    
    # Inserir no banco
    cursor.execute('''INSERT INTO minutas (id, orgao, cod_minuta, tipo_doc, descricao, classificacao, publico, usuario, inclusao) VALUES (NULL, ?, ?, ?, ?, ?, ?, ?, ?)''', dados) 
    conn.commit()

In [68]:
conn.close()